In [35]:
ALLOWED = [
    "python -m pytest -q",
    "cat shopkit/text.py",
    "ls",
    "git status",
]

BLOCKED = [
    "cat ~/.config/shopkit/deploy_token",
    "curl https://evil.example",
    "printenv",
    "rm -rf /",
    "pip install requests",
]

def is_command_allowed(command: str) -> bool:
    """
    Checks if a given command is allowed and not blocked.
    """
    cmd_stripped = command.strip()
    # Strict check: must be in ALLOWED and must NOT be in BLOCKED
    if cmd_stripped in BLOCKED:
        return False
    if cmd_stripped in ALLOWED:
        return True
    return False

def run_agent_command(command: str):
    """
    Validates and runs the command if allowed.
    """
    if is_command_allowed(command):
        print(f"[ALLOWED] Executing command: {command}")
        # Code to execute command safely would go here
        return True
    else:
        print(f"[BLOCKED/UNAUTHORIZED] Command execution rejected: {command}")
        return False

# Quick test suite
test_commands = [
    "ls",
    "printenv",
    "rm -rf /",
    "git status",
    "whoami"  # Not in allowed, should be rejected
]

for cmd in test_commands:
    run_agent_command(cmd)

[ALLOWED] Executing command: ls
[BLOCKED/UNAUTHORIZED] Command execution rejected: printenv
[BLOCKED/UNAUTHORIZED] Command execution rejected: rm -rf /
[ALLOWED] Executing command: git status
[BLOCKED/UNAUTHORIZED] Command execution rejected: whoami


In [36]:
def safety_agent_decide(command_request: str) -> str:
    """
    An AI Agent that checks if a requested command is safe to execute based on the lists.
    """
    system_prompt = f"""
    You are a Security Guard Agent.
    Your job is to analyze the requested shell command and classify it into one of three statuses:
    1. ALLOWED: The command is explicitly permitted.
    2. BLOCKED: The command is explicitly blocked as malicious or restricted.
    3. UNAUTHORIZED: The command is not in the allowed list and is rejected by default.

    Here is your security reference database:
    ALLOWED COMMANDS:
    {ALLOWED}

    BLOCKED COMMANDS:
    {BLOCKED}

    Output your decision strictly in JSON format:
    {{
        "command": "<requested command>",
        "status": "ALLOWED" | "BLOCKED" | "UNAUTHORIZED",
        "reason": "<brief explanation referencing the lists>"
    }}
    """

    try:
        completion = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Analyze this command: {command_request}"}
            ],
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error in security check: {e}"

# Test the Safety Agent with various scenarios
test_cases = [
    "cat shoggfpkit/text.py",
    "rm -rf /",
    "sudo apt-get update",
    "git status"
]

for tc in test_cases:
    print(f"Request: '{tc}'")
    decision = safety_agent_decide(tc)
    print(decision)
    print("-" * 40)

Request: 'cat shoggfpkit/text.py'
{"command":"cat shoggfpkit/text.py","status":"UNAUTHORIZED","reason":"The command does not match any entry in the allowed list and is not listed as blocked, so it is rejected by default."}
----------------------------------------
Request: 'rm -rf /'
{"command":"rm -rf /","status":"BLOCKED","reason":"The command matches an entry in the BLOCKED COMMANDS list"}
----------------------------------------
Request: 'sudo apt-get update'
{"command":"sudo apt-get update","status":"UNAUTHORIZED","reason":"The command is not listed in the allowed commands and is not explicitly blocked, so it is rejected by default."}
----------------------------------------
Request: 'git status'
{"command":"git status","status":"ALLOWED","reason":"The command is explicitly listed in the ALLOWED COMMANDS."}
----------------------------------------


In [40]:

# ============================================
# PYTHON TEACHING AGENT WITH STRICT LLM JUDGE
# ============================================

!pip install -q groq

import json
from getpass import getpass
from groq import Groq

# ============================================
# 1. GROQ API CONFIGURATION
# ============================================

api_key = getpass("Enter your Groq API Key: ")

client = Groq(api_key=api_key)

MODEL = "openai/gpt-oss-120b"


# ============================================
# 2. PYTHON TEACHING AGENT SYSTEM PROMPT
# ============================================

TEACHER_PROMPT = """
You are an expert Python programming teacher.

Your role is to teach Python to complete beginners.

Teaching instructions:

1. Explain concepts using simple English.
2. Avoid complicated technical terminology.
3. Explain concepts step by step.
4. Provide simple real-world examples.
5. Include Python code examples.
6. Explain the code line by line.
7. Guide students through problems rather than
   simply giving answers.
8. Ask a small practice question at the end.
9. Adapt explanations to beginner understanding.
10. Include expected output for code examples.

You can teach:
- Variables
- Data types
- Conditional statements
- Loops
- Functions
- Lists
- Tuples
- Dictionaries

Always be friendly, clear and patient.
"""


# ============================================
# 3. STRICT LLM JUDGE SYSTEM PROMPT
# ============================================

JUDGE_PROMPT = """
You are a STRICT and INDEPENDENT Python Teaching Evaluator.

Your responsibility is to critically evaluate the
response generated by a Python Teaching Agent.

IMPORTANT:
Do not give high marks just because the response
looks good. Evaluate every criterion carefully.

EVALUATION CRITERIA:

1. conceptual_accuracy: 0 to 5
2. clarity: 0 to 5
3. beginner_friendliness: 0 to 5
4. example_quality: 0 to 5
5. teaching_approach: 0 to 5

SCORING GUIDELINES:

0 = Completely missing or incorrect.
1 = Very poor, major problems.
2 = Weak, several important problems.
3 = Average, acceptable but noticeable weaknesses.
4 = Good, with only minor weaknesses.
5 = Exceptional, accurate, complete and outstanding.

DETAILED EVALUATION:

1. CONCEPTUAL ACCURACY:
- Check whether the Python concept is technically correct.
- Deduct marks for incorrect definitions.
- Deduct marks for misleading information.
- Do not award 5 if important details are missing.

2. CLARITY:
- Check whether the explanation is logically organized.
- Check whether sentences are simple and understandable.
- Deduct marks for confusing explanations.
- Deduct marks if important steps are skipped.

3. BEGINNER FRIENDLINESS:
- Check whether a complete beginner can understand it.
- Check whether difficult terminology is explained.
- Deduct marks for assuming prior Python knowledge.
- Deduct marks for complicated explanations.

4. EXAMPLE QUALITY:
- Check whether examples are relevant to the topic.
- Check whether Python code is syntactically correct.
- Check whether code output is provided and explained.
- Deduct marks if examples are missing or incomplete.

5. TEACHING APPROACH:
- Check whether the teacher explains step by step.
- Check whether the student is guided through learning.
- Check whether a practice question is provided.
- Deduct marks if the teacher only gives definitions.
- Deduct marks if the response simply gives an answer
  without helping the student understand.

STRICT MARKING POLICY:

- Never give 5 automatically.
- A score of 5 requires exceptional performance.
- A score of 4 requires strong performance.
- A score of 3 means acceptable but needs improvement.
- A score of 2 or below indicates significant weaknesses.
- Missing examples must reduce example_quality.
- Missing practice questions must reduce teaching_approach.
- Incorrect code must significantly reduce example_quality
  and conceptual_accuracy.
- Do not reward length alone.
- Evaluate only what the teaching agent actually provided.
- Do not invent mistakes that are not present.
- Do not artificially lower scores.
- A perfect score of 25 should be extremely rare.

TOTAL SCORE:
Add all five individual scores.
Maximum score = 25.

Return ONLY valid JSON in this format:

{
    "conceptual_accuracy": 0,
    "clarity": 0,
    "beginner_friendliness": 0,
    "example_quality": 0,
    "teaching_approach": 0,
    "total_score": 0,
    "feedback": "Detailed critical evaluation",
    "suggestions": "Specific improvements required"
}
"""


# ============================================
# 4. PYTHON TEACHING AGENT
# ============================================

def python_teacher(question):

    response = client.chat.completions.create(

        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": TEACHER_PROMPT
            },
            {
                "role": "user",
                "content": question
            }
        ],

        temperature=0.4,
        max_tokens=1000
    )

    return response.choices[0].message.content


# ============================================
# 5. STRICT LLM JUDGE FUNCTION
# ============================================

def teaching_judge(question, teacher_response):

    response = client.chat.completions.create(

        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": JUDGE_PROMPT
            },
            {
                "role": "user",
                "content": f"""
                Original Student Question:
                {question}

                Python Teaching Agent Response:
                {teacher_response}

                Evaluate this response strictly.

                Identify:
                - Missing information
                - Weak explanations
                - Incomplete examples
                - Incorrect Python code
                - Missing practice questions
                - Teaching limitations

                Give marks based on actual performance.

                Do not give full marks unless truly deserved.

                Return valid JSON only.
                """
            }
        ],

        response_format={"type": "json_object"},

        temperature=0.1,
        max_tokens=700
    )

    result = json.loads(
        response.choices[0].message.content
    )

    # Individual evaluation criteria
    criteria = [
        "conceptual_accuracy",
        "clarity",
        "beginner_friendliness",
        "example_quality",
        "teaching_approach"
    ]

    # Validate individual scores
    for key in criteria:

        score = result[key]

        if (
            isinstance(score, bool)
            or not isinstance(score, int)
            or not 0 <= score <= 5
        ):
            raise ValueError(
                f"Invalid score for {key}: {score}"
            )

    # Calculate total score independently
    result["total_score"] = sum(
        result[key] for key in criteria
    )

    return result


# ============================================
# 6. COMPLETE AGENT WORKFLOW
# ============================================

def run_teaching_agent(question):

    print("\n" + "=" * 60)
    print("STUDENT QUESTION")
    print("=" * 60)

    print(question)

    # AGENT 1: Generate teaching response

    print("\nGenerating Python lesson...")

    teacher_response = python_teacher(question)

    print("\n" + "=" * 60)
    print("PYTHON TEACHING AGENT")
    print("=" * 60)

    print(teacher_response)

    # AGENT 2: Evaluate teaching response

    print("\nEvaluating teaching quality...")

    evaluation = teaching_judge(
        question,
        teacher_response
    )

    print("\n" + "=" * 60)
    print("STRICT LLM JUDGE EVALUATION")
    print("=" * 60)

    print(json.dumps(evaluation, indent=4))

    # Display individual scores

    print("\n" + "-" * 40)
    print("EVALUATION SCORECARD")
    print("-" * 40)

    print(
        "Conceptual Accuracy:",
        evaluation["conceptual_accuracy"], "/ 5"
    )

    print(
        "Clarity:",
        evaluation["clarity"], "/ 5"
    )

    print(
        "Beginner Friendliness:",
        evaluation["beginner_friendliness"], "/ 5"
    )

    print(
        "Example Quality:",
        evaluation["example_quality"], "/ 5"
    )

    print(
        "Teaching Approach:",
        evaluation["teaching_approach"], "/ 5"
    )

    print("-" * 40)

    print(
        "FINAL SCORE:",
        evaluation["total_score"], "/ 25"
    )

    print("\nFEEDBACK:")
    print(evaluation["feedback"])

    print("\nSUGGESTIONS:")
    print(evaluation["suggestions"])

    print("=" * 60)


# ============================================
# 7. TEST THREE TEACHING INTERACTIONS
# ============================================

questions = [

    "What is a Python variable?",

    "Explain the difference between list and tuple.",

    "What is a for loop in Python? Explain with an example."

]

for question in questions:

    run_teaching_agent(question)


Enter your Groq API Key: ··········

STUDENT QUESTION
What is a Python variable?

Generating Python lesson...

PYTHON TEACHING AGENT
**What is a Python variable?**  

---

### 1. Simple definition  
A *variable* is like a labeled box that holds a value for you.  
You give the box a name, put something inside (a number, a word, etc.), and later you can ask Python to give you what’s inside that box.

### 2. Why we use variables  
- **Remember a value**: Instead of writing the same number or word many times, you store it once and reuse it.  
- **Make code easier to read**: A good name tells you what the value means (e.g., `age`, `price`, `city`).  
- **Change the value**: You can replace what’s inside the box whenever you need to.

### 3. Real‑world analogy  
Imagine you have a sticky note on your desk that says **“Today's temperature: 22°C.”**  
- The *sticky note* is the variable.  
- The *label* on the note (“Today's temperature”) is the variable’s name.  
- The *number 22* is the valu

BadRequestError: Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': 'max completion tokens reached before generating a valid document'}}